# **get raw data from data lake (s3) into bronze layer**

In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable as T

In [0]:
%run /Workspace/fmcg_pipeline/setup_fmcg_data/utilities


In [0]:
print(bronze_schema)

bronze


In [0]:
dbutils.widgets.text("catalog", "lakehouse", "Catalog")
dbutils.widgets.text("datasource", "customers", "Datasource")
catlog =dbutils.widgets.get("catalog")
dataSource = dbutils.widgets.get("datasource")

## data source is a folder name ofs3 customer data , if folder name change we just change it in a widget


In [0]:
print(catlog)
print(dataSource)

lakehouse
customers


Get data customer folder from s3

In [0]:
path = f's3://fmcg-child-data/{dataSource}/*.csv'
df_bronze = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path).select("*","_metadata.file_name","_metadata.file_size")
df_bronze = df_bronze.withColumn("timestamp", F.current_timestamp())

#df_bronze.printSchema()
#df_bronze.show()


In [0]:
print(path)

s3://fmcg-child-data/customers/*.csv


In [0]:
display(df_bronze.limit(10))

customer_id,customer_name,city,file_name,file_size,timestamp
789201,FitFuel Market,Bengaluru,customers.csv,1404,2026-02-28T22:54:47.514Z
789202,FitFuel Market,Hyderabad,customers.csv,1404,2026-02-28T22:54:47.514Z
789203,FitFuel Market,New Delhi,customers.csv,1404,2026-02-28T22:54:47.514Z
789301,Athlete's Choice Store,Bengaluru,customers.csv,1404,2026-02-28T22:54:47.514Z
789303,Athlete's Choice Store,New Delhi,customers.csv,1404,2026-02-28T22:54:47.514Z
789101,Endurance Foods,Bengalore,customers.csv,1404,2026-02-28T22:54:47.514Z
789102,Endurance Foods,Hyderabad,customers.csv,1404,2026-02-28T22:54:47.514Z
789103,Endurance Foods,New Delhi,customers.csv,1404,2026-02-28T22:54:47.514Z
789121,HydroBoost Nutrition,Hyderabad,customers.csv,1404,2026-02-28T22:54:47.514Z
789122,HydroBoost Nutrition,New Delhi,customers.csv,1404,2026-02-28T22:54:47.514Z


write the data asset in bronze schema

In [0]:
df_bronze.write.format("delta")\
.mode("append")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{bronze_schema}.{dataSource}")

In [0]:
%sql 
SELECT * FROM lakehouse.bronze.customers

customer_id,customer_name,city,file_name,file_size,timestamp
789201,FitFuel Market,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z
789202,FitFuel Market,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
789203,FitFuel Market,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
789301,Athlete's Choice Store,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z
789303,Athlete's Choice Store,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
789101,Endurance Foods,Bengalore,customers.csv,1404,2026-02-28T22:54:54.004Z
789102,Endurance Foods,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
789103,Endurance Foods,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
789121,HydroBoost Nutrition,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
789122,HydroBoost Nutrition,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z


# **processing in silver layer**

In [0]:
%sql
select * from lakehouse.bronze.customers where city is null

customer_id,customer_name,city,file_name,file_size,timestamp
789403,SprintX Nutrition,null,customers.csv,1404,2026-02-28T22:54:54.004Z
789420,ZenAthlete foods,null,customers.csv,1404,2026-02-28T22:54:54.004Z
789521,PrimeFuel Nutrition,null,customers.csv,1404,2026-02-28T22:54:54.004Z
789603,Recovery Lane,null,customers.csv,1404,2026-02-28T22:54:54.004Z
789603,Recovery Lane,null,customers.csv,1404,2026-02-28T22:54:54.004Z


In [0]:
from pyspark.sql import functions as F

# Read bronze table
df_silver = spark.table(f"{catlog}.{bronze_schema}.{dataSource}")

# Identify string columns
string_cols = [f.name for f in df_silver.schema.fields if f.dataType.simpleString() == "string"]

# Trim + normalize spaces
for col in string_cols:
    df_silver = df_silver.withColumn(
        col,
        F.regexp_replace(F.trim(F.col(col)), r"\s+", " ")
    )

# Format customer names
df_silver = df_silver.withColumn("customer_name", F.initcap("customer_name"))

# Fix city typos
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

df_silver = df_silver.replace(city_mapping, subset=["city"])

# Allow only valid cities
allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = df_silver.withColumn(
    "city",
    F.when(F.col("city").isin(allowed), F.col("city"))
)

# Customers whose city may be null
customers = [
    "Recovery Lane",
    "Sprintx Nutrition",
    "Zenathlete Foods",
    "Primefuel Nutrition"
]

customers_df = df_silver.filter(F.col("customer_name").isin(customers))

# Keep rows where city is not null
customers_clean = (
    customers_df
    .dropna(subset=["city"])
    .groupBy("customer_name")
    .agg(F.first("city").alias("city_fill"))
)

# Left join to fill missing cities
joined_df = df_silver.alias("a").join(
    customers_clean.alias("b"),
    on="customer_name",
    how="left"
)

# Replace null city values
result_df = joined_df.withColumn(
    "city",
    F.coalesce(F.col("a.city"), F.col("b.city_fill"))
).drop("city_fill")

df_silver = result_df

# Remove duplicates
df_silver = df_silver.dropDuplicates()
df_silver = (
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    df_silver.withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

# Write to silver delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(f"{catlog}.{silver_schema}.{dataSource}")

customer_name,customer_id,city,file_name,file_size,timestamp
Fitfuel Market,789201,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z
Fitfuel Market,789202,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
Fitfuel Market,789203,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
Athlete's Choice Store,789301,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z
Athlete's Choice Store,789303,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
Endurance Foods,789101,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z
Endurance Foods,789102,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
Endurance Foods,789103,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z
Hydroboost Nutrition,789121,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z
Hydroboost Nutrition,789122,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z


In [0]:
%sql
select * from lakehouse.silver.customers

customer_name,customer_id,city,file_name,file_size,timestamp,customer,market,platform,channel
Sprintx Nutrition,789402,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z,Sprintx Nutrition-Hyderabad,India,Sports Bar,Acquisition
Endurance Foods,789102,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
Hydroboost Nutrition,789122,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
Primefuel Nutrition,789520,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z,Primefuel Nutrition-Bengaluru,India,Sports Bar,Acquisition
Staminax Store,789702,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z,Staminax Store-Hyderabad,India,Sports Bar,Acquisition
Endurance Foods,789103,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
Recovery Lane,789601,Bengaluru,customers.csv,1404,2026-02-28T22:54:54.004Z,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
Staminax Store,789703,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z,Staminax Store-New Delhi,India,Sports Bar,Acquisition
Fitfuel Market,789203,New Delhi,customers.csv,1404,2026-02-28T22:54:54.004Z,Fitfuel Market-New Delhi,India,Sports Bar,Acquisition
Macrobite Superfoods,789221,Hyderabad,customers.csv,1404,2026-02-28T22:54:54.004Z,Macrobite Superfoods-Hyderabad,India,Sports Bar,Acquisition


# **Gold Layer**

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catlog}.{silver_schema}.{dataSource};")

# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catlog}.{gold_schema}.sb_dim_{dataSource}")

In [0]:
%sql
select * from lakehouse.gold.sb_dim_customers;

customer_id,customer_name,city,customer,market,platform,channel
789402,Sprintx Nutrition,Hyderabad,Sprintx Nutrition-Hyderabad,India,Sports Bar,Acquisition
789102,Endurance Foods,Hyderabad,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789520,Primefuel Nutrition,Bengaluru,Primefuel Nutrition-Bengaluru,India,Sports Bar,Acquisition
789702,Staminax Store,Hyderabad,Staminax Store-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods,New Delhi,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789703,Staminax Store,New Delhi,Staminax Store-New Delhi,India,Sports Bar,Acquisition
789203,Fitfuel Market,New Delhi,Fitfuel Market-New Delhi,India,Sports Bar,Acquisition
789221,Macrobite Superfoods,Hyderabad,Macrobite Superfoods-Hyderabad,India,Sports Bar,Acquisition


# **Merging datasource with parent company**

In [0]:
delta_table = T.forName(spark, "lakehouse.gold.dim_customers")
df_child_customers = spark.table("lakehouse.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
df_child_customers = df_child_customers.dropDuplicates(["customer_code"])

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from lakehouse.gold.dim_customers

customer_code,customer,market,platform,channel
789402,Sprintx Nutrition-Hyderabad,India,Sports Bar,Acquisition
789102,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789520,Primefuel Nutrition-Bengaluru,India,Sports Bar,Acquisition
789702,Staminax Store-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789703,Staminax Store-New Delhi,India,Sports Bar,Acquisition
789203,Fitfuel Market-New Delhi,India,Sports Bar,Acquisition
789221,Macrobite Superfoods-Hyderabad,India,Sports Bar,Acquisition
